# Câu 1. Tiền xử lý dữ liệu

In [1]:

import os
import re
import unicodedata
import emoji
from underthesea import word_tokenize

## Đọc dữ liệu

In [2]:
path_in = "../Data1"                  
path_out = "../Data_Preprocessing"    
os.makedirs(path_out, exist_ok=True)
List_Files = sorted([f for f in os.listdir(path_in) if f.endswith(".txt")])
print("Số file dữ liệu:", len(List_Files))


Số file dữ liệu: 172


In [3]:
# 1. Chuẩn hóa văn bản (Standardizing Text)
def text_std(text):
    text = re.sub(r"\s+([\u0300-\u0323])", r"\1", text)  
    text = re.sub(r"([\u0300-\u0323])\s+(?=[aiouycmnpt])([aiouy]?(?:ch|nh|ng|[cmnpt])?)(?!\w)", r"\1\2", text, flags=re.IGNORECASE)
    text = unicodedata.normalize("NFC", text)               
    text = text.replace("\xa0", " ")                        
    return text

# 2. Chuyển văn bản thành chữ thường
def to_lower(text):
    return text.lower()

# 3. Loại bỏ URL
def remove_url(text):
    return re.sub(r"https?://\S+|www\.\S+", " ", text)

# 4. Chuyển biểu tượng cảm xúc thành văn bản
from emot.emo_unicode import UNICODE_EMOJI # For emojis
from emot.emo_unicode import EMOTICONS_EMO # For EMOTICONS
# Function for converting emojis into word
def converting_emojis(text):
    for x in sorted(EMOTICONS_EMO, key=len, reverse=True):   
        text = text.replace(x, " " + "_".join(EMOTICONS_EMO[x].replace(",","").replace(":","").split()).lower() + " ")

    for x in sorted(UNICODE_EMOJI, key=len, reverse=True):
        text = text.replace(x, " " + "_".join(UNICODE_EMOJI[x].replace(",","").replace(":","").split()).lower() + " ")
    return text

# 5. Tách câu
def split_sentences(text):
    List_Sentences = re.split(r"[.!?]+(?=\s|$)|\n+", text)
    return [s.strip() for s in List_Sentences if s.strip() != ""]

# 6. Xóa dấu câu
def remove_punctuation(text):
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(text.split())

# 7. Xóa chữ số
def remove_number(text):
    text = re.sub(r"\w*\d\w*", " ", text)
    return " ".join(text.split())

### 8. Sửa lỗi chính tả 

In [4]:
Dict_TeenCode = {"ko": "không", "k": "không", "kh": "không", "khong": "không", "hok": "không", "hem": "không",
                 "đc": "được", "dc": "được", "đươc": "được", "nv": "nhân viên", "ng": "người", "mn": "mọi người",
                 "j": "gì", "vs": "với", "bt": "bình thường", "r": "rồi", "lm": "làm", "qá": "quá", "wá": "quá",
                 "ntn": "như thế nào", "thik": "thích", "iu": "yêu", "trc": "trước", "sp": "sản phẩm",
                 "a": "anh", "e": "em", "oke": "ok", "okie": "ok", "okela": "ok", "ngta": "người ta", "cx": "cũng"}

def correct_spelling(text):
    text = re.sub(r"(\w)\1{2,}", r"\1", text)   
    words = text.split()
    text_pre = ""
    for word in words:
        if word in Dict_TeenCode:
            word = Dict_TeenCode[word]
        text_pre = text_pre + " " + word
    return text_pre.strip()


### 9. Tách từ

In [5]:
def tokenizing(text):
    List_Words = []
    chunk = []    
    for w in text.split():
        if "_" in w:
            if len(chunk) > 0:
                List_Words.append(word_tokenize(" ".join(chunk), format="text"))
                chunk = []
            List_Words.append(w)     
        else:
            chunk.append(w)
    if len(chunk) > 0:
        List_Words.append(word_tokenize(" ".join(chunk), format="text"))
    return " ".join(List_Words)


### 10. Xóa các từ không có nghĩa 

In [6]:
f = open("vietnamese-stopwords.txt", "r", encoding="utf-8")
List_StopWords = [w.strip().replace(" ", "_") for w in f.read().split("\n") if w.strip() != ""]
f.close()
print("Số stop words:", len(List_StopWords))

def remove_stopwords(text):
    return " ".join(word for word in text.split() if word not in List_StopWords)



Số stop words: 1942


In [7]:
def preprocessing(text):
    text = text_std(text)              
    text = to_lower(text)              
    text = remove_url(text)            
    text = converting_emojis(text)         
    List_Sentences = split_sentences(text)   
    List_Result = []
    for s in List_Sentences:
        s = remove_punctuation(s)      
        s = remove_number(s)           
        s = correct_spelling(s)        
        s = tokenizing(s)              
        s = remove_stopwords(s)        
        if s != "":
            List_Result.append(s)
    return List_Result

for file_name in List_Files:
    f = open(os.path.join(path_in, file_name), "r", encoding="utf-8")
    text = f.read()
    f.close()

    List_Result = preprocessing(text)

    f = open(os.path.join(path_out, file_name), "w", encoding="utf-8")
    f.write("\n".join(List_Result))
    f.close()

print("Đã tiền xử lý và lưu", len(os.listdir(path_out)), "file vào", path_out)

Đã tiền xử lý và lưu 172 file vào ../Data_Preprocessing


## Kiểm tra kết quả

In [8]:
for file_name in List_Files[:10]:
    print("=====", file_name, "=====")
    print("[Trước]:", open(os.path.join(path_in, file_name), "r", encoding="utf-8").read())
    print("[Sau]:")
    print(open(os.path.join(path_out, file_name), "r", encoding="utf-8").read())
    print()


===== 11509.txt =====
[Trước]: Uô ́ ng ơ ̉ đây cu ̃ ng nhiê ̀ u lâ ̀ n , đê ̀ u la ̀ take away . Nhân_viên không nơ ̉ nô ̉ i 1 nu ̣ cươ ̀ i vơ ́ i kha ́ ch ma ̀ mă ̣ t cư ́ la ̣ nh tanh . Thêm 1 ca ́ i nư ̃ a la ̀ nghe nhâ ̀ m order ( không biê ́ t co ́ pha ̉ i mi ̀ nh mi ̀ nh bi ̣ không ) . Bi ̣ mâ ́ y lâ ̀ n , lâ ̀ n đâ ̀ u co ̀ n kêu đô ̉ i chư ́ mâ ́ y lâ ̀ n sau thiê ̣ t pha ́ t bư ̣ c qua ́ nên thôi luôn . Đươ ̣ c ca ́ i tra ̀ sư ̃ a ngon , hơi ngo ̣ t , nhiê ̀ u đa ́ .

[Sau]:
uống take away
nhân_viên nở nổi nụ cười mặt lạnh_tanh
nhầm order
mấy đầu kêu đổi mấy thiệt_phát bực
trà sữa ngon hơi đá

===== 11516.txt =====
[Trước]: Kéo vào quán do không biết đi đâu , món nướng lại làm sẵn , thiếu đồ , gọi 3 4 lần mà nhân_viên không đem nhưng khi thanh_toán lại có trong bill .

[Sau]:
kéo quán đi món nướng sẵn đồ gọi nhân_viên đem thanh_toán bill

===== 11517.txt =====
[Trước]: Chỉ có 2 chữ dành cho quán " TỒI_TỆ " . Thật_sự mình không biết dùng từ gì cho cái quán này nữa . Hồi đó mình